# 02 — Análisis v2: tendencias, géneros, desarrolladoras y valor

**Input:** `data/clean/steam.db`  
**Output:** gráficos en `img/`

Cubre las 4 líneas de análisis de v2:
1. Tendencias por año de lanzamiento
2. Comparativa entre géneros: precio, playtime y reviews
3. Ranking de desarrolladoras por review score histórico
4. Métrica compuesta de valor

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

DB_PATH = '../data/clean/steam.db'

def query(sql):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

---
## 1. Tendencias por año de lanzamiento

¿Cuántos juegos se lanzaron por año? ¿El review score promedio mejoró o empeoró con el tiempo? ¿El precio subió?

In [ ]:
df_anio = query("""
    SELECT
        release_year,
        COUNT(*) AS lanzamientos,
        ROUND(AVG(review_score), 1) AS review_promedio,
        ROUND(AVG(CASE WHEN price > 0 THEN price END), 2) AS precio_promedio,
        ROUND(AVG(average_playtime), 0) AS playtime_promedio
    FROM games
    WHERE release_year BETWEEN 2000 AND 2019
      AND review_score IS NOT NULL
      AND total_ratings >= 50
    GROUP BY release_year
    ORDER BY release_year
""")

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].bar(df_anio['release_year'], df_anio['lanzamientos'], color='#5b8db8', alpha=0.85)
axes[0].set_ylabel('Cantidad de juegos')
axes[0].set_title('Lanzamientos por año')

axes[1].plot(df_anio['release_year'], df_anio['review_promedio'],
             color='#4a9e6b', marker='o', linewidth=2)
axes[1].set_ylabel('Review score (%)')
axes[1].set_ylim(50, 100)
axes[1].set_title('Review score promedio por año')
axes[1].axhline(df_anio['review_promedio'].mean(), color='gray',
                linestyle='--', alpha=0.6, label='Promedio general')
axes[1].legend()

axes[2].plot(df_anio['release_year'], df_anio['precio_promedio'],
             color='#e07b39', marker='s', linewidth=2)
axes[2].set_ylabel('Precio promedio (USD)')
axes[2].set_xlabel('Año')
axes[2].set_title('Precio promedio de juegos de pago por año')

plt.tight_layout()
plt.savefig('../img/02_tendencias_por_anio.png', dpi=150)
plt.show()
df_anio

---
## 2. Comparativa entre géneros

Comparamos precio, playtime y review score por género. Solo géneros simples con suficiente muestra.

In [ ]:
df_gen = query("""
    SELECT
        genres,
        COUNT(*) AS total_juegos,
        ROUND(AVG(review_score), 1) AS review_promedio,
        ROUND(AVG(CASE WHEN price > 0 THEN price END), 2) AS precio_promedio,
        ROUND(AVG(average_playtime), 0) AS playtime_promedio,
        ROUND(AVG(horas_por_dolar), 1) AS horas_dolar_promedio
    FROM games
    WHERE review_score IS NOT NULL
      AND total_ratings >= 100
      AND genres NOT LIKE '%,%'
    GROUP BY genres
    HAVING total_juegos >= 40
    ORDER BY review_promedio DESC
    LIMIT 12
""")

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
generos = df_gen['genres']

axes[0].barh(generos, df_gen['review_promedio'], color='#5b8db8', alpha=0.85)
axes[0].set_xlabel('Review score promedio (%)')
axes[0].set_title('Review score')
axes[0].set_xlim(50, 100)
for i, v in enumerate(df_gen['review_promedio']):
    axes[0].text(v + 0.3, i, f'{v}%', va='center', fontsize=9)

axes[1].barh(generos, df_gen['precio_promedio'], color='#e07b39', alpha=0.85)
axes[1].set_xlabel('Precio promedio (USD)')
axes[1].set_title('Precio promedio')
axes[1].set_yticklabels([])

axes[2].barh(generos, df_gen['playtime_promedio'], color='#4a9e6b', alpha=0.85)
axes[2].set_xlabel('Playtime promedio (min)')
axes[2].set_title('Tiempo de juego promedio')
axes[2].set_yticklabels([])

plt.suptitle('Comparativa por género: review, precio y playtime', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../img/02_comparativa_generos.png', dpi=150, bbox_inches='tight')
plt.show()
df_gen

In [ ]:
# Heatmap de correlación entre métricas por género
corr = df_gen[['review_promedio', 'precio_promedio', 'playtime_promedio', 'horas_dolar_promedio']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=['Review', 'Precio', 'Playtime', 'Hs/$'],
            yticklabels=['Review', 'Precio', 'Playtime', 'Hs/$'], ax=ax)
ax.set_title('Correlación entre métricas por género')
plt.tight_layout()
plt.savefig('../img/02_correlacion_generos.png', dpi=150)
plt.show()

---
## 3. Ranking de desarrolladoras por review score histórico

Desarrolladoras con al menos 5 juegos publicados, ordenadas por review score promedio.

In [ ]:
df_dev = query("""
    SELECT
        developer,
        COUNT(*) AS juegos_publicados,
        ROUND(AVG(review_score), 1) AS review_promedio,
        SUM(total_ratings) AS ratings_totales,
        ROUND(AVG(CASE WHEN price > 0 THEN price END), 2) AS precio_promedio,
        ROUND(AVG(horas_por_dolar), 1) AS horas_dolar_promedio
    FROM games
    WHERE review_score IS NOT NULL
      AND total_ratings >= 50
      AND developer IS NOT NULL
      AND developer != ''
    GROUP BY developer
    HAVING juegos_publicados >= 5
    ORDER BY review_promedio DESC
    LIMIT 20
""")

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(df_dev['developer'], df_dev['review_promedio'],
               color='#7b68c8', alpha=0.85)
ax.set_xlabel('Review score promedio (%)')
ax.set_title('Top 20 desarrolladoras por review score histórico (mín. 5 juegos)')
ax.set_xlim(60, 105)
for bar, (_, row) in zip(bars, df_dev.iterrows()):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['review_promedio']}% · {int(row['juegos_publicados'])} juegos",
            va='center', fontsize=8.5)
plt.tight_layout()
plt.savefig('../img/02_ranking_desarrolladoras.png', dpi=150)
plt.show()
df_dev

In [ ]:
# Scatter: volumen de juegos vs review score
df_dev_all = query("""
    SELECT developer,
           COUNT(*) AS juegos,
           ROUND(AVG(review_score), 1) AS review_promedio,
           SUM(total_ratings) AS ratings_totales
    FROM games
    WHERE review_score IS NOT NULL AND total_ratings >= 50
      AND developer IS NOT NULL AND developer != ''
    GROUP BY developer
    HAVING juegos >= 3
""")

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    df_dev_all['juegos'],
    df_dev_all['review_promedio'],
    s=df_dev_all['ratings_totales'].clip(0, 100000) / 500,
    alpha=0.4, color='#7b68c8'
)
ax.set_xlabel('Cantidad de juegos publicados')
ax.set_ylabel('Review score promedio (%)')
ax.set_title('Volumen de juegos vs calidad percibida por desarrolladora')
ax.set_ylim(30, 100)
plt.tight_layout()
plt.savefig('../img/02_volumen_vs_calidad.png', dpi=150)
plt.show()

---
## 4. Métrica compuesta de valor

`valor_score` fue calculada en el notebook 01 combinando review score (50%), horas por dólar (35%) y popularidad (15%). Acá analizamos su distribución y qué juegos y géneros tienen el mayor score.

In [ ]:
# Top 25 juegos por valor_score
df_top_valor = query("""
    SELECT name, developer, genres,
           ROUND(price, 2) AS price,
           ROUND(review_score, 1) AS review_score,
           ROUND(horas_por_dolar, 1) AS horas_por_dolar,
           total_ratings,
           ROUND(valor_score, 1) AS valor_score
    FROM games
    WHERE valor_score IS NOT NULL
    ORDER BY valor_score DESC
    LIMIT 25
""")

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(df_top_valor['name'].str[:45], df_top_valor['valor_score'],
               color='#e07b39', alpha=0.85)
ax.set_xlabel('Valor score (0–100)')
ax.set_title('Top 25 juegos por métrica compuesta de valor')
ax.set_xlim(0, 110)
for bar, val in zip(bars, df_top_valor['valor_score']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../img/02_top_valor_score.png', dpi=150)
plt.show()
df_top_valor

In [ ]:
# Valor score promedio por categoría de precio
df_valor_precio = query("""
    SELECT categoria_precio,
           COUNT(*) AS juegos,
           ROUND(AVG(valor_score), 1) AS valor_promedio,
           ROUND(AVG(review_score), 1) AS review_promedio
    FROM games
    WHERE valor_score IS NOT NULL
      AND categoria_precio IS NOT NULL
    GROUP BY categoria_precio
    ORDER BY valor_promedio DESC
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_valor_precio['categoria_precio'], df_valor_precio['valor_promedio'],
       color=['#5b8db8', '#4a9e6b', '#e07b39', '#7b68c8', '#d4695a'], alpha=0.85)
ax.set_ylabel('Valor score promedio')
ax.set_xlabel('Categoría de precio')
ax.set_title('¿El precio más alto garantiza más valor?')
for i, (_, row) in enumerate(df_valor_precio.iterrows()):
    ax.text(i, row['valor_promedio'] + 0.5, f"{row['valor_promedio']}",
            ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../img/02_valor_por_precio.png', dpi=150)
plt.show()
df_valor_precio

In [ ]:
# Valor score promedio por género
df_valor_gen = query("""
    SELECT genres,
           COUNT(*) AS juegos,
           ROUND(AVG(valor_score), 1) AS valor_promedio
    FROM games
    WHERE valor_score IS NOT NULL
      AND genres NOT LIKE '%,%'
    GROUP BY genres
    HAVING juegos >= 20
    ORDER BY valor_promedio DESC
    LIMIT 12
""")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_valor_gen['genres'], df_valor_gen['valor_promedio'],
        color='#4a9e6b', alpha=0.85)
ax.set_xlabel('Valor score promedio (0–100)')
ax.set_title('Valor score promedio por género')
for i, v in enumerate(df_valor_gen['valor_promedio']):
    ax.text(v + 0.3, i, f'{v}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../img/02_valor_por_genero.png', dpi=150)
plt.show()

---
## Resumen de hallazgos v2

*(Completar con los valores reales al ejecutar)*

| Análisis | Hallazgo |
|---|---|
| Tendencias por año | TBD — ¿cuál fue el año con más lanzamientos? ¿review score subió o bajó? |
| Mejor género por reviews | TBD |
| Género con más playtime | TBD |
| Mejor desarrolladora histórica | TBD |
| Top juego por valor_score | TBD |
| ¿Precio alto = más valor? | TBD — hipótesis a confirmar con los datos |